# EPR-based Quantum Key Distribution

In the EPR QKD protocol, Alice and Bob share entangled pairs (EPR pairs). Each party independently chooses a random measurement basis. When their bases match, their results are perfectly correlated and can be used as key bits.

An eavesdropper measuring Bob's qubit before him introduces detectable errors.

In [ ]:
import qsharp
import matplotlib.pyplot as plt
import random

In [ ]:
qsharp.init()
with open("Program.qs") as f:
    qsharp.eval(f.read())

from qsharp.code import EPRPrepareAndMeasure, EPRWithEavesdropper

In [ ]:
def run_epr_qkd(n_pairs, eavesdropper_prob=0.0):
    alice_bases = [random.choice([True, False]) for _ in range(n_pairs)]
    bob_bases = [random.choice([True, False]) for _ in range(n_pairs)]

    alice_results = []
    bob_results = []
    for i in range(n_pairs):
        if eavesdropper_prob > 0.0 and random.random() < eavesdropper_prob:
            eve_basis = random.choice([True, False])
            r = EPRWithEavesdropper(alice_bases[i], bob_bases[i], eve_basis)
        else:
            r = EPRPrepareAndMeasure(alice_bases[i], bob_bases[i])
        alice_results.append(str(r[0]) == "One")
        bob_results.append(str(r[1]) == "One")

    # Sifting: keep only matching bases
    alice_key = []
    bob_key = []
    for i in range(n_pairs):
        if alice_bases[i] == bob_bases[i]:
            alice_key.append(alice_results[i])
            bob_key.append(bob_results[i])

    # Eavesdropper check
    check_bits = []
    final_alice = []
    final_bob = []
    for i in range(len(alice_key)):
        if random.random() < 0.5:
            check_bits.append(alice_key[i] != bob_key[i])
        else:
            final_alice.append(alice_key[i])
            final_bob.append(bob_key[i])

    error_rate = sum(check_bits) / len(check_bits) if check_bits else 0.0
    keys_match = final_alice == final_bob

    return {
        "key_length": len(final_alice),
        "error_rate": error_rate,
        "keys_match": keys_match,
    }

In [ ]:
scenarios = [
    ("No eavesdropper", 0.0),
    ("50% eavesdropping", 0.5),
    ("100% eavesdropping", 1.0),
]

error_rates = []
for name, prob in scenarios:
    result = run_epr_qkd(512, eavesdropper_prob=prob)
    error_rates.append(result["error_rate"])
    status = "MATCH" if result["keys_match"] else "MISMATCH"
    detected = "Eavesdropper detected!" if result["error_rate"] > 0 else "No eavesdropper"
    print(f"{name}: error rate = {result['error_rate']*100:.1f}%, key length = {result['key_length']}, keys {status}. {detected}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([s[0] for s in scenarios], [r * 100 for r in error_rates])
ax.set_ylabel("Error rate (%)")
ax.set_ylim(0, 30)
plt.title("EPR QKD - Error Rate vs Eavesdropping")
plt.show()